In [ ]:
 # Auto-detect runtime, set env, and launch the Semi-DETR script from Jupyter

import os, sys, subprocess, shlex, json, pathlib, time

SCRIPT = "TrainSemiDETR_SOLO_SingleClass_STATIC_Ucloud.py"

def _detect_runtime_profile() -> str:
    forced = os.environ.get("RFDETR_RUNTIME_PROFILE", "auto").strip().lower()
    if forced in {"ucloud", "local"}:
        return forced
    if forced not in {"", "auto"}:
        raise ValueError("RFDETR_RUNTIME_PROFILE must be one of: auto, ucloud, local")
    if os.name != "nt" and pathlib.Path("/work").exists():
        has_member_files = any(pathlib.Path("/work").glob("Member Files:*"))
        has_hash_user = any(pathlib.Path("/work").glob("*#*"))
        if has_member_files or has_hash_user:
            return "ucloud"
    return "local"

RUNTIME_PROFILE = _detect_runtime_profile()
os.environ["RFDETR_RUNTIME_PROFILE"] = RUNTIME_PROFILE
if RUNTIME_PROFILE == "ucloud":
    PROJECT_DIR = os.environ.get("PROJECT_DIR", "/work/projects/myproj/SOLO_SemiSupervised_RFDETR/")
else:
    PROJECT_DIR = os.environ.get("PROJECT_DIR", str(pathlib.Path.cwd() / "SOLO_SemiSupervised_RFDETR"))
print("RFDETR_RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("PROJECT_DIR =", PROJECT_DIR)

PROJECT_PATH = pathlib.Path(PROJECT_DIR).resolve()
REPO_ROOT = PROJECT_PATH.parent

_dataset_root_candidates = [
    PROJECT_PATH / "Stat_Dataset",
    REPO_ROOT / "SOLO_Supervised_RFDETR" / "Stat_Dataset",
    REPO_ROOT / "Stat_Dataset",
]
_dataset_root_auto = next((p for p in _dataset_root_candidates if p.exists()), _dataset_root_candidates[0])
DATASET_ROOT = pathlib.Path(os.environ.get("STAT_DATASETS_ROOT", str(_dataset_root_auto))).resolve()

def _latest_dataset_dir(root: pathlib.Path, token: str) -> str:
    cands = sorted([p for p in root.glob(f"*{token}*") if p.is_dir()])
    return str(cands[-1]) if cands else ""

os.environ["DATASET_EPI"] = os.environ.get("DATASET_EPI", "").strip() or _latest_dataset_dir(DATASET_ROOT, "SquamousEpithelialCell_OVR")
os.environ["DATASET_LEUCO"] = os.environ.get("DATASET_LEUCO", "").strip() or _latest_dataset_dir(DATASET_ROOT, "Leucocyte_OVR")

if RUNTIME_PROFILE == "ucloud":
    def _detect_ucloud_user_base() -> str:
        work = pathlib.Path("/work")
        if not work.exists():
            return ""
        member = sorted(work.glob("Member Files:*"))
        if member:
            return member[0].name
        hashed = sorted([p for p in work.glob("*#*") if p.is_dir()])
        return hashed[0].name if hashed else ""

    UCLOUD_USER_BASE = os.environ.get("USER_BASE_DIR", "").strip() or _detect_ucloud_user_base()
    if UCLOUD_USER_BASE:
        IMAGES_FALLBACK_DEFAULT = f"/work/{UCLOUD_USER_BASE}/CellScanData/Zoom10x - Quality Assessment_Cleaned"
        OUTPUT_ROOT_DEFAULT = f"/work/{UCLOUD_USER_BASE}/SemiDETR_OUTPUT"
    else:
        IMAGES_FALLBACK_DEFAULT = str(REPO_ROOT / "CellScanData" / "Zoom10x - Quality Assessment_Cleaned")
        OUTPUT_ROOT_DEFAULT = str(PROJECT_PATH / "SemiDETR_OUTPUT")
else:
    IMAGES_FALLBACK_DEFAULT = str(pathlib.Path(r"D:\PHD\PhdData\CellScanData\Zoom10x - Quality Assessment_Cleaned"))
    OUTPUT_ROOT_DEFAULT = str(PROJECT_PATH / "SemiDETR_OUTPUT_local")

os.environ.setdefault("IMAGES_FALLBACK_ROOT", IMAGES_FALLBACK_DEFAULT)
os.environ.setdefault("SEMIDETR_OUTPUT_ROOT", OUTPUT_ROOT_DEFAULT)

# Resolve Semi-DETR repo path on both local and UCloud; clone if missing/invalid.
_semidetr_expected_rel = pathlib.Path("configs") / "detr_ssod" / "detr_ssod_dino_detr_r50_coco_120k.py"

def _repo_ok(p: pathlib.Path) -> bool:
    return p.exists() and (p / _semidetr_expected_rel).exists() and (p / "tools" / "train_detr_ssod.py").exists()

_semidetr_repo_env = os.environ.get("SEMIDETR_REPO_DIR", "").strip()
_semidetr_repo_candidates = []
if _semidetr_repo_env:
    _semidetr_repo_candidates.append(pathlib.Path(_semidetr_repo_env).expanduser())
_semidetr_repo_candidates.extend([
    REPO_ROOT / "_external_SemiDETR_ref",
    PROJECT_PATH / "_external_SemiDETR_ref",
    pathlib.Path("/work/projects/myproj/_external_SemiDETR_ref"),
])

_semidetr_repo = next((p for p in _semidetr_repo_candidates if _repo_ok(p)), None)

if _semidetr_repo is None:
    auto_clone = os.environ.get("SEMIDETR_AUTO_CLONE", "1").strip().lower() in {"1", "true", "yes", "on"}
    if not auto_clone:
        raise FileNotFoundError(
            "Could not find a valid Semi-DETR repo (missing configs/tools). "
            "Set SEMIDETR_REPO_DIR to a full clone or enable SEMIDETR_AUTO_CLONE=1."
        )

    clone_target_txt = os.environ.get("SEMIDETR_CLONE_TARGET", "").strip()
    clone_target = pathlib.Path(clone_target_txt).expanduser() if clone_target_txt else (REPO_ROOT / "_external_SemiDETR_ref_runtime")

    # If target exists but is not a valid clone (e.g. empty gitlink placeholder), avoid using it directly.
    if clone_target.exists() and not _repo_ok(clone_target):
        if any(clone_target.iterdir()):
            clone_target = REPO_ROOT / f"_external_SemiDETR_ref_runtime_{int(time.time())}"
        else:
            clone_target.rmdir()

    print(f"[SETUP] Cloning official Semi-DETR repo to {clone_target} ...")
    subprocess.run(["git", "clone", "https://github.com/JCZ404/Semi-DETR", str(clone_target)], check=True)
    _semidetr_repo = clone_target

if not _repo_ok(_semidetr_repo):
    raise FileNotFoundError(
        f"Resolved SEMIDETR_REPO_DIR is invalid: {_semidetr_repo} (missing {_semidetr_expected_rel})"
    )

os.environ["SEMIDETR_REPO_DIR"] = str(_semidetr_repo)
os.environ["SEMIDETR_BASE_CONFIG"] = str(pathlib.Path(os.environ["SEMIDETR_REPO_DIR"]) / _semidetr_expected_rel)

# Experiment setup
os.environ["SEMIDETR_TARGET"] = "epi"
os.environ["SEMIDETR_INIT_MODE"] = "scratch"
os.environ["SEMIDETR_TRAIN_FRACTIONS"] = "1.0,0.5"
os.environ["SEMIDETR_SEEDS"] = "42"
os.environ["SEMIDETR_FRACTION_SEEDS"] = "42"

# Semi-DETR protocol knobs
os.environ.setdefault("SEMIDETR_MAX_ITERS", "120000")
os.environ.setdefault("SEMIDETR_EVAL_INTERVAL", "4000")
os.environ.setdefault("SEMIDETR_CKPT_INTERVAL", "4000")
os.environ.setdefault("SEMIDETR_SAMPLES_PER_GPU", "5")
os.environ.setdefault("SEMIDETR_WORKERS_PER_GPU", "5")
os.environ.setdefault("SEMIDETR_SAMPLE_RATIO", "1,4")
os.environ.setdefault("SEMIDETR_PSEUDO_SCORE_THRESH", "0.5")
os.environ.setdefault("SEMIDETR_UNSUP_WEIGHT", "4.0")
os.environ.setdefault("SEMIDETR_UNLABELED_IMAGES_PER_LABELED", "6")
os.environ.setdefault("SEMIDETR_UNLABELED_MAX_IMAGES", "0")

# Preflight
for _name, _p in (
    ("DATASET_EPI", os.environ.get("DATASET_EPI", "")),
    ("DATASET_LEUCO", os.environ.get("DATASET_LEUCO", "")),
    ("IMAGES_FALLBACK_ROOT", os.environ.get("IMAGES_FALLBACK_ROOT", "")),
    ("SEMIDETR_REPO_DIR", os.environ.get("SEMIDETR_REPO_DIR", "")),
    ("SEMIDETR_BASE_CONFIG", os.environ.get("SEMIDETR_BASE_CONFIG", "")),
):
    if not str(_p).strip():
        raise ValueError(f"{_name} is empty")
    if not pathlib.Path(_p).exists():
        raise FileNotFoundError(f"Configured path does not exist for {_name}: {_p}")

print("SEMIDETR_TARGET =", os.environ["SEMIDETR_TARGET"])
print("SEMIDETR_INIT_MODE =", os.environ["SEMIDETR_INIT_MODE"])
print("SEMIDETR_TRAIN_FRACTIONS =", os.environ["SEMIDETR_TRAIN_FRACTIONS"])
print("SEMIDETR_SEEDS =", os.environ["SEMIDETR_SEEDS"])
print("SEMIDETR_FRACTION_SEEDS =", os.environ["SEMIDETR_FRACTION_SEEDS"])
print("SEMIDETR_MAX_ITERS =", os.environ["SEMIDETR_MAX_ITERS"])
print("SEMIDETR_SAMPLE_RATIO =", os.environ["SEMIDETR_SAMPLE_RATIO"])
print("SEMIDETR_PSEUDO_SCORE_THRESH =", os.environ["SEMIDETR_PSEUDO_SCORE_THRESH"])
print("SEMIDETR_UNSUP_WEIGHT =", os.environ["SEMIDETR_UNSUP_WEIGHT"])
print("DATASET_EPI =", os.environ["DATASET_EPI"])
print("DATASET_LEUCO =", os.environ["DATASET_LEUCO"])
print("IMAGES_FALLBACK_ROOT =", os.environ["IMAGES_FALLBACK_ROOT"])
print("SEMIDETR_REPO_DIR =", os.environ["SEMIDETR_REPO_DIR"])
print("SEMIDETR_BASE_CONFIG =", os.environ["SEMIDETR_BASE_CONFIG"])
print("SEMIDETR_OUTPUT_ROOT =", os.environ["SEMIDETR_OUTPUT_ROOT"])

# ---- Run the script ----
wd = pathlib.Path(PROJECT_DIR).resolve()
script_path = wd / SCRIPT
if not wd.exists():
    raise NotADirectoryError(f"PROJECT_DIR does not exist: {wd}")
if not script_path.exists():
    raise FileNotFoundError(f"Training script not found: {script_path}")

py = sys.executable
cmd = [py, "-u", str(script_path)]

print("\n[LAUNCH]")
print(" cwd:", wd)
print(" cmd:", " ".join(shlex.quote(x) for x in cmd))

proc = subprocess.Popen(
    cmd,
    cwd=str(wd),
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Run failed with exit code {ret}")
